Step 1: IMPORT SNOWPARK AND THEN TRUNCATE THE ALL 3 TABLES IN STAGING BEFORE LOADING THE FILE DATA INTO STAGING TABLES 

In [ ]:
import snowflake.snowpark as snowpark

In [ ]:
# Use SQL to set the current database and schema
session.sql('USE SCHEMA SNOWPARK_DB.STAGING').collect()

# Use SQL to Truncate the India Sales Order table
session.sql('TRUNCATE TABLE SNOWPARK_DB.STAGING.INDIA_SALES_ORDER_CP').collect()

# Use SQL to Truncate the Usa Sales Order table
session.sql('TRUNCATE TABLE SNOWPARK_DB.STAGING.USA_SALES_ORDER_CP').collect()

# Use SQL to Truncate the France Sales Order table
session.sql('TRUNCATE TABLE SNOWPARK_DB.STAGING.FRANCE_SALES_ORDER_CP').collect()

######
#Step 2 - Load the file data to the respective copy tables
######
# ls snowpark_stage columns 
        #  s3://aj-snowpark-data-bucket/ORDER-Sales-FR.json
        # s3://aj-snowpark-data-bucket/ORDER-Sales-IN.csv
        # s3://aj-snowpark-data-bucket/ORDER-Sales-USA.parquet


# Load the India Sales Order table
session.sql("""COPY INTO SNOWPARK_DB.STAGING.INDIA_SALES_ORDER_CP
               FROM @SNOWPARK_DB.STAGING.SNOWPARK_STAGE/ORDER-Sales-IN.csv 
               FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"')""").collect()

# Load the Usa Sales Order table
session.sql("""COPY INTO SNOWPARK_DB.STAGING.USA_SALES_ORDER_CP
               FROM @SNOWPARK_DB.STAGING.SNOWPARK_STAGE/ORDER-Sales-USA.parquet
               FILE_FORMAT = (TYPE = PARQUET)""").collect()

# Load the France Sales Order table
session.sql("""COPY INTO SNOWPARK_DB.STAGING.FRANCE_SALES_ORDER_CP 
               FROM @SNOWPARK_DB.STAGING.SNOWPARK_STAGE/ORDER-Sales-FR.json 
               FILE_FORMAT = (TYPE = JSON)""").collect()

#### step 4 : Loading the data from the copy tables to raw tables 

# Create the India Sales Order raw table dataframe
df_india_sales_read = session.sql("""SELECT
                                ORDER_ID,
                                CUSTOMER_NAME,
                                MOBILE_MODEL,
                                QUANTITY,
                                PRICE_PER_UNIT,
                                TOTAL_PRICE,
                                PROMOTION_CODE,
                                ORDER_AMOUNT,
                                GST,
                                ORDER_DATE,
                                PAYMENT_STATUS,
                                SHIPPING_STATUS,
                                PAYMENT_METHOD,
                                PAYMENT_PROVIDER,
                                MOBILE,
                                DELIVERY_ADDRESS,
                                CURRENT_TIMESTAMP() AS INSERT_DTS
                                FROM SNOWPARK_DB.STAGING.INDIA_SALES_ORDER_CP""")

# Save above Dataframe as a India Sales order raw table
df_india_sales_read.write.mode("overwrite").save_as_table("SNOWPARK_DB.RAW.INDIA_SALES_ORDER")


# Create the USA Sales Order raw table dataframe
df_usa_sales_read = session.sql("""SELECT 
                                SOURCE_DATA:"Customer Name"::VARCHAR(1000) AS CUSTOMER_NAME,
                                SOURCE_DATA:"Delivery Address"::VARCHAR(1000) AS DELIVERY_ADDRESS,
                                SOURCE_DATA:"Mobile Model"::VARCHAR(1000) AS MOBILE_MODEL,
                                SOURCE_DATA:"Order Amount"::VARCHAR(1000) AS ORDER_AMOUNT,
                                SOURCE_DATA:"Order Date"::VARCHAR(1000) AS ORDER_DATE,
                                SOURCE_DATA:"Order ID"::VARCHAR(1000) AS ORDER_ID,
                                SOURCE_DATA:"Payment Method"::VARCHAR(1000) AS PAYMENT_METHOD,
                                SOURCE_DATA:"Payment Provider"::VARCHAR(1000) AS PAYMENT_PROVIDER,
                                SOURCE_DATA:"Payment Status"::VARCHAR(1000) AS PAYMENT_STATUS,
                                SOURCE_DATA:"Phone"::VARCHAR(1000) AS PHONE,
                                SOURCE_DATA:"Price per Unit"::VARCHAR(1000) AS PRICE_PER_UNIT,
                                SOURCE_DATA:"Promotion Code"::VARCHAR(1000) AS PROMOTION_CODE,
                                SOURCE_DATA:"Quantity"::VARCHAR(1000) AS QUANTITY,
                                SOURCE_DATA:"Shipping Status"::VARCHAR(1000) AS SHIPPING_STATUS,
                                SOURCE_DATA:"Tax"::VARCHAR(1000) AS TAX,
                                SOURCE_DATA:"Total Price"::VARCHAR(1000) AS TOTAL_PRICE,
                                CURRENT_TIMESTAMP() AS INSERT_DTS
                                FROM SNOWPARK_DB.STAGING.USA_SALES_ORDER_CP""")

# Save above Dataframe as a USA Sales order raw table
df_usa_sales_read.write.mode("overwrite").save_as_table("SNOWPARK_DB.RAW.USA_SALES_ORDER")

# Create the France Sales Order raw table dataframe
df_france_sales_read = session.sql("""SELECT 
                                B.VALUE:"Customer Name"::VARCHAR(1000) AS CUSTOMER_NAME,
                                B.VALUE:"Delivery Address"::VARCHAR(1000) AS DELIVERY_ADDRESS,
                                B.VALUE:"Mobile Model"::VARCHAR(1000) AS MOBILE_MODEL,
                                B.VALUE:"Order Amount"::VARCHAR(1000) AS ORDER_AMOUNT,
                                B.VALUE:"Order Date"::VARCHAR(1000) AS ORDER_DATE,
                                B.VALUE:"Order ID"::VARCHAR(1000) AS ORDER_ID,
                                B.VALUE:"Payment Method"::VARCHAR(1000) AS PAYMENT_METHOD,
                                B.VALUE:"Payment Provider"::VARCHAR(1000) AS PAYMENT_PROVIDER,
                                B.VALUE:"Payment Status"::VARCHAR(1000) AS PAYMENT_STATUS,
                                B.VALUE:"Phone"::VARCHAR(1000) AS PHONE,
                                B.VALUE:"Price per Unit"::VARCHAR(1000) AS PRICE_PER_UNIT,
                                B.VALUE:"Promotion Code"::VARCHAR(1000) AS PROMOTION_CODE,
                                B.VALUE:"Quantity"::VARCHAR(1000) AS QUANTITY,
                                B.VALUE:"Shipping Status"::VARCHAR(1000) AS SHIPPING_STATUS,
                                B.VALUE:"Tax"::VARCHAR(1000) AS TAX,
                                B.VALUE:"Total Price"::VARCHAR(1000) AS TOTAL_PRICE,
                                CURRENT_TIMESTAMP AS INSERT_DTS
                                FROM SNOWPARK_DB.STAGING.FRANCE_SALES_ORDER_CP A,
                                LATERAL FLATTEN (INPUT=> A.SOURCE_DATA) B""")

# Save above Dataframe as a France Sales order raw table
df_france_sales_read.write.mode("overwrite").save_as_table("SNOWPARK_DB.RAW.FRANCE_SALES_ORDER")


print("Successfully Executed the Snowpark Code to Load Raw tables from Staging Layer")

Now Transformation code for data in the raw tables and then moving it to transformed layer

Transformation 1: Order columns in all the raw tables in sequential order, rename (ex;  TAX, GST, MOBILE, Comtact )  and add new columns (Country)
Transformation 2 : Union all the 3 raw tables into one df 
Transformation 3: Data Validation and Checks - Missing values with default values 
Transforamtion 4 : Split one column into multiple columns ( Mobile Brand and Model )

In [ ]:
session.sql('USE SCHEMA SNOWPARK_DB.RAW').collect()

In [ ]:
from snowflake.snowpark.functions import col,lit,split, current_timestamp

df_india_sales_order= session.sql("""SELECT
                                ORDER_ID   ,
                            	CUSTOMER_NAME   ,
                            	MOBILE_MODEL   ,
                            	QUANTITY   ,
                            	PRICE_PER_UNIT   ,
                            	TOTAL_PRICE   ,
                            	PROMOTION_CODE   ,
                            	ORDER_AMOUNT   ,
                            	GST   ,
                            	ORDER_DATE   ,
                            	PAYMENT_STATUS   ,
                            	SHIPPING_STATUS   ,
                            	PAYMENT_METHOD   ,
                            	PAYMENT_PROVIDER   ,
                            	MOBILE   ,
                            	DELIVERY_ADDRESS   ,
                            	FROM SNOWPARK_DB.RAW.INDIA_SALES_ORDER""")

# RENAMING THE GST AND MOBILE COLUMNS 
df_india_sales_order_renamed=df_india_sales_order.rename("GST","TAX").rename("MOBILE","CONTACT_NUMBER")

# ADD COUNTRY FIELD using lit() - literal function for fixed value of country

df_india_sales_order_country=df_india_sales_order_renamed.withColumn("Country",lit("INDIA"))



In [ ]:
df_usa_sales_order=session.sql("""SELECT 
                                ORDER_ID,
                                CUSTOMER_NAME,  
                                MOBILE_MODEL,
                                QUANTITY,
                                PRICE_PER_UNIT,
                                TOTAL_PRICE,
                                PROMOTION_CODE,
                                ORDER_AMOUNT,
                                TAX,
                                ORDER_DATE,
                                PAYMENT_STATUS,
                                SHIPPING_STATUS,
                                PAYMENT_METHOD,
                                PAYMENT_PROVIDER,
                                PHONE,
                                DELIVERY_ADDRESS
                                FROM SNOWPARK_DB.RAW.USA_SALES_ORDER""")

# RENAME PHONE TO CONTACT_NUMBER

df_usa_sales_order_renamed=df_usa_sales_order.rename("PHONE","CONTACT_NUMBER")

df_usa_sales_order_country=df_usa_sales_order_renamed.withColumn("COUNTRY",lit("USA"))
            

In [ ]:
df_france_sales_order=session.sql("""SELECT 
                                ORDER_ID,
                                CUSTOMER_NAME,  
                                MOBILE_MODEL,
                                QUANTITY,
                                PRICE_PER_UNIT,
                                TOTAL_PRICE,
                                PROMOTION_CODE,
                                ORDER_AMOUNT,
                                TAX,
                                ORDER_DATE,
                                PAYMENT_STATUS,
                                SHIPPING_STATUS,
                                PAYMENT_METHOD,
                                PAYMENT_PROVIDER,
                                PHONE,
                                DELIVERY_ADDRESS
                                FROM SNOWPARK_DB.RAW.FRANCE_SALES_ORDER""")

df_france_sales_order_renamed=df_france_sales_order.rename("PHONE","CONTACT_NUMBER")

df_france_sales_order_country=df_france_sales_order_renamed.withColumn("COUNTRY",lit("FRANCE"))

In [ ]:
df_india_usa_sales_order=df_india_sales_order_country.union(df_usa_sales_order_country)
df_india_usa_france_sales_order=df_india_usa_sales_order.union(df_france_sales_order_country)

In [ ]:
df_india_usa_france_sales_order.show(5)

In [ ]:
df_india_usa_france_sales_order=df_india_usa_france_sales_order.fillna("NA",subset="promotion_code") # subset can also take a list 

In [ ]:
df_union_sales_order_split=df_india_usa_france_sales_order\
    .withColumn("MOBILE_BRAND",   split(col("MOBILE_MODEL"), lit("/")).getItem(0).cast("string")) \
    .withColumn("MOBILE_VERSION", split(col("MOBILE_MODEL"), lit("/")).getItem(1).cast("string")) \
    .withColumn("MOBILE_COLOR",   split(col("MOBILE_MODEL"), lit("/")).getItem(2).cast("string")) \
    .withColumn("MOBILE_RAM",     split(col("MOBILE_MODEL"), lit("/")).getItem(3).cast("string")) \
    .withColumn("MOBILE_MEMORY",  split(col("MOBILE_MODEL"), lit("/")).getItem(4).cast("string"))

Rearrange the columns and add the insert dts column in the final df 

In [ ]:
df_union_sales_order_final=df_union_sales_order_split.select(col("ORDER_ID"),
                                                            col("CUSTOMER_NAME"),
                                                            col("MOBILE_BRAND"),
                                                            col("MOBILE_VERSION").alias("MOBILE_MODEL"),
                                                            col("MOBILE_COLOR"),
                                                            col("MOBILE_RAM"),
                                                            col("MOBILE_MEMORY"),
                                                            col("QUANTITY"),
                                                            col("PRICE_PER_UNIT"),
                                                            col("TOTAL_PRICE"),
                                                            col("PROMOTION_CODE"),
                                                            col("ORDER_AMOUNT"),
                                                            col("TAX"),
                                                            col("ORDER_DATE"),
                                                            col("PAYMENT_STATUS"),
                                                            col("SHIPPING_STATUS"),
                                                            col("PAYMENT_METHOD"),
                                                            col("PAYMENT_PROVIDER"),
                                                            col("CONTACT_NUMBER"),
                                                            col("DELIVERY_ADDRESS"),
                                                            col("COUNTRY")).withColumn("INSERT_DTS",current_timestamp())



In [ ]:
df_union_sales_order_final.write.mode("overwrite").save_as_table("SNOWPARK_DB.TRANSFORMED.GLOBAL_SALES_ORDER")

For Curated Schema 

1. Table with only delivered shipping status 
2. Sum of sales by mobile brand 

In [ ]:
session.sql('USE SCHEMA SNOWPARK_DB.TRANSFORMED').collect()

In [ ]:
# Global Sales Order delivered
df_global_sales_order_delivered = session.table("SNOWPARK_DB.TRANSFORMED.GLOBAL_SALES_ORDER").\
                                       filter(col("SHIPPING_STATUS") == 'Delivered')

# Load the data into the table
df_global_sales_order_delivered.write.mode("overwrite").save_as_table("SNOWPARK_DB.CURATED.GLOBAL_SALES_ORDER_DELIVERED")

Global sales table by aggregating the sales based on mobile brand and model model and renaming total price total sum to total sales_amount


In [ ]:
from snowflake.snowpark.functions import sum as sum_

# Global Sales Order brand
df_global_sales_order_brand = session.table("SNOWPARK_DB.TRANSFORMED.GLOBAL_SALES_ORDER").\
                                   groupBy(col("MOBILE_BRAND"),col("MOBILE_MODEL")).\
                                   agg(sum_(col("TOTAL_PRICE")).alias("TOTAL_SALES_AMOUNT"))

# Load the data into the table
df_global_sales_order_brand.write.mode("overwrite").save_as_table("SNOWPARK_DB.CURATED.GLOBAL_SALES_ORDER_BRAND")

global sales table by aggregating the sales based on country , generate year and quarter , caluclate volumne and total sales 


In [ ]:
# Global Sales Order country
from snowflake.snowpark.functions import col, sum, year, quarter, to_date, concat, lit

df_global_sales_order_country = session.table("SNOWPARK_DB.TRANSFORMED.GLOBAL_SALES_ORDER").\
                                       groupBy(col("COUNTRY"),
                                               year(to_date(col("ORDER_DATE"))).alias("YEAR"),
                                               concat(lit("Q"),quarter(to_date(col("ORDER_DATE")))).alias("QUARTER")).\
                                       agg(sum(col("QUANTITY")).alias("TOTAL_SALES_VOLUME"),
                                          sum(col("TOTAL_PRICE")).alias("TOTAL_SALES_AMOUNT"))

# Load the data into the table
df_global_sales_order_country.write.mode("overwrite").save_as_table("SNOWPARK_DB.CURATED.GLOBAL_SALES_ORDER_COUNTRY")
